In [1]:
pip install pyspark numpy jupyter matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# ============================================
# SE446 - Milestone 2: Spark Analytics (Phase A)
# Group: Altamimi-AlDeri
#
# Tasks 1, 2, 3, 4: Abdulaziz Altamimi (ID: 230714)
# ============================================
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Must be set before the JVM starts — fixes Windows "Python worker timeout" error
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ["_JAVA_OPTIONS"] = "-Djava.net.preferIPv4Stack=true"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, when,
    round as spark_round
)

# Auto-detect environment: cluster vs local
ON_CLUSTER = (
    os.environ.get('YARN_CONF_DIR') is not None or
    os.environ.get('HADOOP_CONF_DIR') is not None or
    os.path.exists('/data/chicago_crimes.csv')
)

mode = 'CLUSTER' if ON_CLUSTER else 'LOCAL'
print(f'Running in {mode} mode')
print(f'Python executable: {sys.executable}')

if ON_CLUSTER:
    spark = (
        SparkSession.builder
        .appName('SE446-M2-PhaseA-Altamimi-AlDeri')
        .getOrCreate()
    )
else:
    spark = (
        SparkSession.builder
        .appName('SE446-M2-PhaseA-Altamimi-AlDeri')
        .master('local[*]')
        .config('spark.driver.memory', '2g')
        .config('spark.sql.shuffle.partitions', '4')
        .config('spark.pyspark.python', sys.executable)
        .config('spark.pyspark.driver.python', sys.executable)
        .getOrCreate()
    )

spark.sparkContext.setLogLevel('WARN')
os.makedirs('output', exist_ok=True)
print(f'Spark version: {spark.version}')
print(f'Web UI: {spark.sparkContext.uiWebUrl}')

Running in LOCAL mode
Python executable: c:\Users\Admin\python\python.exe
Spark version: 4.1.2
Web UI: http://WINDOWS-ISFVGL7.mshome.net:4040


In [16]:
# ============================================
# Data Loading / Generation
# Author: Abdulaziz Altamimi (ID: 230714)
# ============================================

import random
import pandas as pd

if ON_CLUSTER:
    DATA_PATH = 'hdfs:///data/chicago_crimes.csv'
    print(f'Loading data from HDFS: {DATA_PATH}')
    df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
else:
    # ---- LOCAL MODE: Generate realistic sample data ----
    from pyspark.sql import Row
    import random
    random.seed(42)

    crime_profiles = {
        "NARCOTICS":         0.85,
        "PROSTITUTION":      0.80,
        "WEAPONS VIOLATION": 0.60,
        "BATTERY":           0.30,
        "ASSAULT":           0.25,
        "ROBBERY":           0.15,
        "THEFT":             0.10,
        "BURGLARY":          0.08,
        "MOTOR VEHICLE THEFT": 0.06,
        "CRIMINAL DAMAGE":   0.05,
    }
    districts = list(range(1, 26))

    LOCATIONS = [
        "STREET", "RESIDENCE", "APARTMENT", "SIDEWALK", "OTHER",
        "PARKING LOT / GARAGE(NON.RESID.)", "ALLEY",
        "SCHOOL, PUBLIC, BUILDING", "SMALL RETAIL STORE", "RESTAURANT"
    ]
    def generate_row():
        crime_type = random.choice(list(crime_profiles.keys()))
        base_rate = crime_profiles[crime_type]
        district = random.choice(districts)
        hour_val = random.randint(0, 23)
        domestic = random.random() < 0.15
        year = random.randint(2001, 2023)
        location=random.choice(LOCATIONS)
        arrest_prob = base_rate + (0.20 if domestic else 0)
        if 2 <= hour_val <= 5:
            arrest_prob -= 0.10
        arrest_prob = max(0.01, min(0.99, arrest_prob))
        arrest = random.random() < arrest_prob
        return Row(
            District=district, PrimaryType=crime_type,
            Hour=hour_val, Domestic_str=str(domestic).lower(),
            Arrest=arrest, label=int(arrest), Year=year, 
            LocationDescription=location
        )

    rows = [generate_row() for _ in range(10000)]
    df = spark.createDataFrame(rows)

if ON_CLUSTER:
    total = df.count()
else:
    try:
        total = len(rows)
    except NameError:
        total = df.count()
print(f'Total records: {total:,}')
print('Schema:')
df.printSchema()
df.show(5, truncate=False)


Total records: 10,000
Schema:
root
 |-- District: long (nullable = true)
 |-- PrimaryType: string (nullable = true)
 |-- Hour: long (nullable = true)
 |-- Domestic_str: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- label: long (nullable = true)
 |-- Year: long (nullable = true)
 |-- LocationDescription: string (nullable = true)

+--------+-------------------+----+------------+------+-----+----+-------------------+
|District|PrimaryType        |Hour|Domestic_str|Arrest|label|Year|LocationDescription|
+--------+-------------------+----+------------+------+-----+----+-------------------+
|1       |PROSTITUTION       |23  |false       |true  |1    |2008|APARTMENT          |
|3       |MOTOR VEHICLE THEFT|18  |false       |false |0    |2001|RESIDENCE          |
|20      |MOTOR VEHICLE THEFT|0   |false       |false |0    |2023|SMALL RETAIL STORE |
|19      |BURGLARY           |8   |false       |false |0    |2001|APARTMENT          |
|9       |ROBBERY            |4   |fa

---
## Phase A: Spark DataFrame Analytics


---
### Task 1: Crime Type Distribution (Spark DataFrame)

**Goal**: Count crimes by `Primary Type`, show top 10 ordered by count descending.  
**Author**: Abdulaziz Altamimi (ID: 230714)


In [17]:
# ============================================
# Task 1: Crime Type Distribution
# Author: Abdulaziz Altamimi (ID: 230714)
# ============================================
print('TASK 1: Crime Type Distribution (Spark DataFrame)')
print()
print('Top 10 Crime Types by Count:')

crime_dist = (
    df.groupBy('PrimaryType')
      .count()
      .orderBy(col('count').desc())
).show(10, truncate=False)


TASK 1: Crime Type Distribution (Spark DataFrame)

Top 10 Crime Types by Count:
+-------------------+-----+
|PrimaryType        |count|
+-------------------+-----+
|WEAPONS VIOLATION  |1065 |
|THEFT              |1054 |
|MOTOR VEHICLE THEFT|1005 |
|PROSTITUTION       |1005 |
|ASSAULT            |998  |
|NARCOTICS          |991  |
|BATTERY            |987  |
|BURGLARY           |976  |
|ROBBERY            |973  |
|CRIMINAL DAMAGE    |946  |
+-------------------+-----+



In [12]:
# Stop Spark session when done
spark.stop()
print('SparkSession stopped.')


SparkSession stopped.
